# 🛠️ Tensorbox Setup & Kaggle Dataset Ingestion

Welcome to **Tensorbox**! This setup notebook configures your environment credentials and downloads the **authentic Kaggle benchmark datasets** required across all 14 curriculum tracks and 10 enterprise projects.

---
### 📋 What this notebook does:
1. **Configures Credentials**: Prompts for your Kaggle credentials (and optional LLM keys) and persists them securely to `.env`.
2. **Ingests Kaggle Datasets**: Downloads all authentic Kaggle datasets directly into the `data/` directory.
3. **Workspace Diagnostics**: Verifies GPU acceleration, directories, and database services.

## 🔑 Step 1: Configure Environment & Kaggle Credentials
Set your Kaggle credentials below. In Jupyter, you can enter them in the form or update your `.env` directly.

In [ ]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from utils.config import Config
from utils.logger import get_logger

logger = get_logger("SetupNotebook")
print(f"Tensorbox Workspace Root: {ROOT_DIR}")
print(f"Active Python: {sys.executable}")

In [ ]:
print("=" * 60)
print("🔐 KAGGLE & LLM CREDENTIALS SETUP")
print("=" * 60)

# To set new keys interactively, edit these strings or pass them via environment
kaggle_user = os.environ.get("KAGGLE_USERNAME", Config.get("KAGGLE_USERNAME", ""))
kaggle_key = os.environ.get("KAGGLE_KEY", Config.get("KAGGLE_KEY", ""))

if kaggle_user:
    Config.set("KAGGLE_USERNAME", kaggle_user)
    print(f"✓ KAGGLE_USERNAME: {kaggle_user}")
else:
    print("ℹ️ KAGGLE_USERNAME not set (will use authentic Kaggle mirror downloads)")

if kaggle_key:
    Config.set("KAGGLE_KEY", kaggle_key)
    print("✓ KAGGLE_KEY: configured securely")
else:
    print("ℹ️ KAGGLE_KEY not set")

print("\nCredentials configuration status ready.")

## 📥 Step 2: Download All Authentic Kaggle Benchmark Datasets
The cell below ingests all authentic Kaggle datasets and caches them directly into the `data/` directory.

In [ ]:
import pandas as pd
from utils.data_loader import KAGGLE_DATASET_REGISTRY, download_kaggle_dataset, get_data_dir

data_dir = get_data_dir()
print(f"Target dataset directory: {data_dir}\n")

summary_records = []
for name, meta in KAGGLE_DATASET_REGISTRY.items():
    if meta["file_type"] != "CSV":
        continue
    target_file = data_dir / meta["filename"]
    print(f"▶ Ingesting {name} ({meta['description']})...")
    success = download_kaggle_dataset(name, target_file)
    
    if target_file.exists():
        df = pd.read_csv(target_file)
        summary_records.append({
            "Dataset": name,
            "Kaggle Slug": meta["kaggle_slug"],
            "File Size": f"{target_file.stat().st_size / 1024:.1f} KB",
            "Rows": len(df),
            "Columns": len(df.columns),
            "Status": "✅ Ingested"
        })
    else:
        summary_records.append({
            "Dataset": name,
            "Kaggle Slug": meta["kaggle_slug"],
            "File Size": "N/A",
            "Rows": 0,
            "Columns": 0,
            "Status": "❌ Failed"
        })

summary_df = pd.DataFrame(summary_records)
print("\nIngestion Summary:")
print(summary_df.to_string(index=False))

## 🩺 Step 3: Workspace Health Diagnostics & Readiness Verification

In [ ]:
import torch
from utils.health_check import check_all_services

print("=" * 60)
print("🩺 WORKSPACE HEALTH CHECK")
print("=" * 60)

gpu_avail = torch.cuda.is_available() or (hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
device_name = "CUDA GPU" if torch.cuda.is_available() else ("Apple Silicon MPS" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "CPU")
print(f"• Acceleration Hardware: {device_name} (Active: {gpu_avail})")

report = check_all_services()
for srv, st in report.get("services", {}).items():
    icon = "🟢" if st.get("status") == "online" else "⚪"
    print(f"{icon} Service {srv:<15}: {st.get('status', 'unknown')} ({st.get('details', '')})")

print("\n🎉 Setup is 100% complete! You can now proceed to any Curriculum Track or Project.")